# Module 1: Setup and Base Model Evaluation

**Duration:** ~20 minutes

## Learning Objectives
- Set up the Google Colab environment with GPU support
- Install required dependencies
- Load TinyLlama-1.1B with 4-bit quantization
- Evaluate baseline model performance on F5 technical questions

## Overview
In this module, we'll set up our environment and establish a baseline for how a general-purpose LLM performs on F5-specific technical questions. This baseline will be compared against RAG-enhanced and fine-tuned models in later modules.

## 1.1 Environment Check

First, let's verify we have GPU access. If you see "No GPU", go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU detected!")
    print("   Go to Runtime > Change runtime type > T4 GPU")

## 1.2 Install Dependencies

We'll install the core libraries needed for this lab. This may take a few minutes.

In [ ]:
%%capture
# Install core dependencies
!pip install -q torch transformers accelerate
!pip install -q bitsandbytes>=0.49.0
!pip install -q peft>=0.18.0
!pip install -q datasets jsonlines

# For Colab, we need to handle some package issues
!pip install -q sentencepiece protobuf

In [ ]:
# Verify installations
import transformers
import peft
import bitsandbytes

print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print(f"torch: {torch.__version__}")
print("\n✅ All packages installed successfully!")

## 1.3 Clone the Lab Repository

Let's get the training data and documentation files.

In [ ]:
import os

# Clone the repository (update URL as needed)
if not os.path.exists('LLM-FineTuning-RAG-Lab'):
    !git clone https://github.com/yourusername/LLM-FineTuning-RAG-Lab.git
    print("✅ Repository cloned!")
else:
    print("ℹ️ Repository already exists")

# Change to the lab directory
os.chdir('LLM-FineTuning-RAG-Lab')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Verify data files exist
print("📁 Data files:")
print("\nF5 Documentation:")
!ls -la data/f5_docs/

print("\nTraining Data:")
!ls -la data/training/

## 1.4 Load the Base Model

We'll use TinyLlama-1.1B-Chat with 4-bit quantization. This reduces the model size from ~4GB to ~2GB while maintaining reasonable quality.

### Why TinyLlama?
- **Small enough** for Colab free tier (~2GB VRAM with 4-bit)
- **Chat-optimized** for instruction following
- **Good baseline** to demonstrate enhancement techniques

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Model configuration
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False
)

print(f"Loading {MODEL_NAME}...")
print("This may take 1-2 minutes for the first download.")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Tokenizer loaded")
print(f"   Vocabulary size: {tokenizer.vocab_size:,}")

In [ ]:
# Load model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"\n✅ Model loaded successfully!")
print(f"   Model type: {type(model).__name__}")
print(f"   Device: {model.device}")

In [ ]:
# Check GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU Memory:")
    print(f"   Allocated: {allocated:.2f} GB")
    print(f"   Reserved: {reserved:.2f} GB")

## 1.5 Define the Generation Function

TinyLlama uses a specific chat template. We'll create a helper function for generating responses.

In [ ]:
def generate_response(question, max_new_tokens=256, temperature=0.7):
    """
    Generate a response from the model using TinyLlama's chat template.
    
    Args:
        question: The user's question
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature (higher = more creative)
    
    Returns:
        The model's response
    """
    # TinyLlama chat format
    prompt = f"""<|system|>
You are an F5 Networks technical expert. Provide accurate, detailed answers about BIG-IP, iRules, load balancing, SSL offloading, and other F5 technologies.</s>
<|user|>
{question}</s>
<|assistant|>
"""
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and extract response
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the assistant's response
    if "<|assistant|>" in full_response:
        response = full_response.split("<|assistant|>")[-1].strip()
    else:
        # Fallback: get text after the question
        response = full_response[len(prompt):].strip()
    
    return response

print("✅ Generation function defined")

## 1.6 Test Basic Generation

Let's make sure the model is working with a simple test.

In [ ]:
# Simple test
test_question = "What is Python?"
print(f"Question: {test_question}\n")
print(f"Response: {generate_response(test_question, max_new_tokens=100)}")

## 1.7 Baseline Evaluation on F5 Questions

Now let's test the model on F5-specific questions to establish our baseline. These questions cover various F5 topics that we'll use for comparison throughout the lab.

In [ ]:
# F5 Technical Questions for Baseline Evaluation
eval_questions = [
    "What is a virtual server in F5 BIG-IP and what is its purpose?",
    "How do I configure SSL offloading on F5 BIG-IP?",
    "Write an iRule to redirect HTTP to HTTPS.",
    "What is the difference between Least Connections and Round Robin load balancing?",
    "How do I troubleshoot a pool member that is marked as offline?"
]

print(f"Evaluating {len(eval_questions)} questions...\n")
print("=" * 80)

In [ ]:
# Store baseline responses
baseline_responses = []

for i, question in enumerate(eval_questions, 1):
    print(f"\n{'='*80}")
    print(f"Question {i}: {question}")
    print("-" * 80)
    
    response = generate_response(question)
    baseline_responses.append({
        "question": question,
        "response": response
    })
    
    print(f"Response:\n{response}")

print(f"\n{'='*80}")
print("Baseline evaluation complete!")

## 1.8 Analysis: Baseline Limitations

Take a moment to review the responses above. Consider:

1. **Accuracy**: Are the technical details correct?
2. **Specificity**: Does it mention F5-specific terminology and features?
3. **Completeness**: Does it fully answer the question?
4. **Actionability**: Could someone follow this advice?

You may notice the baseline model:
- Provides generic networking answers
- Lacks F5-specific terminology (TMSH, iRules, profiles)
- May make up plausible-sounding but incorrect information
- Doesn't reference actual configuration commands

## 1.9 Save Baseline Results

Let's save these results for comparison in Module 4.

In [ ]:
import json
import os

# Create results directory
os.makedirs("results", exist_ok=True)

# Save baseline results
results = {
    "model": MODEL_NAME,
    "type": "baseline",
    "responses": baseline_responses
}

with open("results/baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Baseline results saved to results/baseline_results.json")

## 1.10 Summary

In this module, we:

1. ✅ Set up the Google Colab environment with GPU
2. ✅ Installed required dependencies
3. ✅ Loaded TinyLlama-1.1B with 4-bit quantization
4. ✅ Tested baseline performance on F5 questions
5. ✅ Identified limitations in domain-specific knowledge

### Key Observations
- The base model has general knowledge but lacks F5-specific expertise
- 4-bit quantization allows running a 1.1B model on free GPU resources
- We have a clear baseline to measure improvements against

### Next Steps
In **Module 2**, we'll build a RAG system that augments the model with F5 documentation, allowing it to retrieve and use relevant context when answering questions.

In [ ]:
# Cleanup (optional - free GPU memory if needed)
# Uncomment if you want to free memory before the next module

# import gc
# del model
# del tokenizer
# gc.collect()
# torch.cuda.empty_cache()
# print("GPU memory cleared")